# TabML Spaceship Titanic v12 - Enhanced Performance

This notebook improves upon v11 by incorporating advanced techniques from the baseline notebook:
- Enhanced feature engineering (cabin splitting, group features, spending aggregations)
- Better hyperparameters tuned for performance
- Improved ensemble strategies

Target: Improve from 79.8% to approach baseline's 82.066% accuracy

## 1. Setup and Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# TabML imports
from tabml.feature_manager import feature_manager
from tabml.algorithms import (
    XGBoostAlgorithm,
    LightGBMAlgorithm,
    CatBoostAlgorithm,
    RandomForestAlgorithm,
    ExtraTreesAlgorithm,
    NeuralNetworkAlgorithm,
)
from tabml.ensembles import (
    SimpleAverageEnsemble,
    WeightedAverageEnsemble,
    StackingEnsemble,
    OptunaWeightedEnsemble,
    RankAverageEnsemble,
)
from tabml.pipelines import FeaturePipeline, ModelPipeline
from tabml.utils.utils import get_categorical_columns, get_numerical_columns

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and Basic Data Exploration

In [ ]:
# Load data
train_df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
submission = pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Display basic info
train_df.info()
train_df.head()

## 3. Advanced Feature Engineering

### 3.1 Extract Group and Cabin Components

In [ ]:
def extract_advanced_features(df):
    """Extract advanced features from PassengerId and Cabin"""
    df = df.copy()
    
    # Extract passenger group from PassengerId
    df['Group'] = df['PassengerId'].str.split('_').str[0].astype('Int64')
    df['GroupSize'] = df.groupby('Group')['PassengerId'].transform('count')
    df['PersonInGroup'] = df['PassengerId'].str.split('_').str[1].astype('Int64')
    
    # Extract cabin components
    cabin_split = df['Cabin'].str.split('/', expand=True)
    df['CabinDeck'] = cabin_split[0]
    df['CabinNum'] = pd.to_numeric(cabin_split[1], errors='coerce')
    df['CabinSide'] = cabin_split[2]
    
    # Cabin-based features
    df['CabinRegion'] = df['CabinDeck'].fillna('Unknown') + '_' + df['CabinSide'].fillna('Unknown')
    
    # Family features from name
    df['LastName'] = df['Name'].str.split().str[-1]
    df['FamilySize'] = df.groupby('LastName')['PassengerId'].transform('count')
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    return df

train_df = extract_advanced_features(train_df)
test_df = extract_advanced_features(test_df)

### 3.2 Create Spending Features

In [ ]:
def create_spending_features(df):
    """Create various spending aggregations"""
    df = df.copy()
    
    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # Total spending
    df['TotalSpending'] = df[spending_cols].sum(axis=1)
    
    # Luxury spending (Spa + VRDeck + RoomService)
    df['LuxurySpending'] = df[['Spa', 'VRDeck', 'RoomService']].sum(axis=1)
    
    # Basic spending (FoodCourt + ShoppingMall)
    df['BasicSpending'] = df[['FoodCourt', 'ShoppingMall']].sum(axis=1)
    
    # Spending ratios
    df['LuxuryRatio'] = df['LuxurySpending'] / (df['TotalSpending'] + 1)
    df['SpendingPerAge'] = df['TotalSpending'] / (df['Age'] + 1)
    
    # Binary features
    df['NoSpending'] = (df['TotalSpending'] == 0).astype(int)
    df['HighSpender'] = (df['TotalSpending'] > df['TotalSpending'].quantile(0.75)).astype(int)
    
    # Count of services used
    df['ServicesUsed'] = (df[spending_cols] > 0).sum(axis=1)
    
    return df

train_df = create_spending_features(train_df)
test_df = create_spending_features(test_df)

### 3.3 Handle Missing Values with Advanced Imputation

In [ ]:
def handle_missing_values(train, test):
    """Handle missing values with smart imputation"""
    train = train.copy()
    test = test.copy()
    
    # Handle boolean columns first - CRITICAL FIX
    bool_cols = ['CryoSleep', 'VIP']
    for col in bool_cols:
        if col in train.columns:
            # Convert string representations to boolean
            train[col] = train[col].replace({'True': True, 'False': False, 'true': True, 'false': False})
            test[col] = test[col].replace({'True': True, 'False': False, 'true': True, 'false': False})
            
            # Fill missing with False (most common)
            train[col] = train[col].fillna(False)
            test[col] = test[col].fillna(False)
            
            # Ensure boolean type
            train[col] = train[col].astype(bool)
            test[col] = test[col].astype(bool)
    
    # Impute Age using group/family median
    for df in [train, test]:
        # First try group median
        df['Age'] = df.groupby('Group')['Age'].transform(
            lambda x: x.fillna(x.median()) if x.notna().any() else x
        )
        # Then try family median
        df['Age'] = df.groupby('LastName')['Age'].transform(
            lambda x: x.fillna(x.median()) if x.notna().any() else x
        )
        # Finally use overall median
        df['Age'].fillna(df['Age'].median(), inplace=True)
    
    # Impute spending features
    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in spending_cols:
        # If CryoSleep is True, spending should be 0
        mask_cryo = train['CryoSleep'] == True
        train.loc[mask_cryo, col] = train.loc[mask_cryo, col].fillna(0)
        
        mask_cryo = test['CryoSleep'] == True
        test.loc[mask_cryo, col] = test.loc[mask_cryo, col].fillna(0)
        
        # For others, use median
        train[col].fillna(train[col].median(), inplace=True)
        test[col].fillna(test[col].median(), inplace=True)
    
    # Impute categorical features
    cat_cols = ['HomePlanet', 'Destination', 'CabinDeck', 'CabinSide']
    for col in cat_cols:
        if col in train.columns:
            mode_val = train[col].mode()[0] if not train[col].mode().empty else 'Unknown'
            train[col].fillna(mode_val, inplace=True)
            test[col].fillna(mode_val, inplace=True)
    
    # Fill remaining NaN in numeric columns
    numeric_cols = train.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col != 'Transported':
            train[col].fillna(train[col].median(), inplace=True)
            test[col].fillna(train[col].median(), inplace=True)
    
    # Fill remaining NaN in object columns
    object_cols = train.select_dtypes(include=['object']).columns
    for col in object_cols:
        train[col].fillna('Unknown', inplace=True)
        test[col].fillna('Unknown', inplace=True)
    
    return train, test

train_df, test_df = handle_missing_values(train_df, test_df)

### 3.4 Create Interaction Features

In [ ]:
def create_interaction_features(df):
    """Create interaction and derived features"""
    df = df.copy()
    
    # Age-based features
    df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], 
                            labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])
    df['IsChild'] = (df['Age'] < 18).astype(int)
    df['IsSenior'] = (df['Age'] >= 60).astype(int)
    
    # CryoSleep interactions
    df['CryoAge'] = df['CryoSleep'].astype(int) * df['Age']
    df['CryoVIP'] = df['CryoSleep'].astype(int) * df['VIP'].astype(int)
    
    # Planet-Destination interaction
    df['Route'] = df['HomePlanet'].astype(str) + '_to_' + df['Destination'].astype(str)
    
    # Group characteristics
    df['GroupSpending'] = df.groupby('Group')['TotalSpending'].transform('sum')
    df['GroupAvgAge'] = df.groupby('Group')['Age'].transform('mean')
    df['GroupCryoRate'] = df.groupby('Group')['CryoSleep'].transform('mean')
    
    # Cabin density features
    df['CabinDensity'] = df.groupby(['CabinDeck', 'CabinSide'])['PassengerId'].transform('count')
    
    return df

train_df = create_interaction_features(train_df)
test_df = create_interaction_features(test_df)

## 4. Prepare Features for Modeling

In [ ]:
# Separate features and target
target_col = 'Transported'
id_col = 'PassengerId'

# Convert target to binary
train_df[target_col] = train_df[target_col].astype(int)

# Features to drop
drop_cols = [id_col, target_col, 'Name', 'Cabin', 'LastName']  # Keep processed features

# Get feature columns
feature_cols = [col for col in train_df.columns if col not in drop_cols]

# Prepare X and y
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()
X_test = test_df[feature_cols].copy()

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"\nFeature columns ({len(feature_cols)}):")
print(feature_cols[:20])  # Show first 20 features

## 5. Feature Encoding

In [ ]:
# Identify categorical columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"\nNumerical features ({len(numerical_features)}): {numerical_features[:10]}...")  # Show first 10

# Label encode categorical features
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    
    # Fit on combined data to handle unseen categories
    combined = pd.concat([X_train[col], X_test[col]], axis=0)
    le.fit(combined.astype(str))
    
    # Transform
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    
    label_encoders[col] = le

print("\nEncoding complete!")

## 6. Advanced Model Configuration with Optimized Hyperparameters

In [ ]:
# Create optimized models based on baseline analysis
models = {}

# XGBoost with optimized parameters
models['xgb'] = XGBoostAlgorithm(
    n_estimators=469,
    max_depth=4,
    learning_rate=0.0202,
    subsample=0.7465,
    colsample_bytree=0.8499,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0
)

# LightGBM with optimized parameters
models['lgb'] = LightGBMAlgorithm(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.0077,
    subsample=0.6189,
    colsample_bytree=0.7775,
    reg_alpha=0.1433,
    reg_lambda=0.9310,
    num_leaves=31,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

# CatBoost with optimized parameters
models['cat'] = CatBoostAlgorithm(
    iterations=500,
    depth=5,
    learning_rate=0.01,
    l2_leaf_reg=0.5,
    random_strength=0.1,
    max_bin=100,
    grow_policy='Lossguide',
    bootstrap_type='Bernoulli',
    subsample=0.8,
    one_hot_max_size=10,
    random_seed=RANDOM_STATE,
    verbose=False
)

# CatBoost variant with SymmetricTree
models['cat_sym'] = CatBoostAlgorithm(
    iterations=400,
    depth=6,
    learning_rate=0.015,
    l2_leaf_reg=1.0,
    random_strength=0.2,
    max_bin=150,
    grow_policy='SymmetricTree',
    bootstrap_type='MVS',
    subsample=0.85,
    random_seed=RANDOM_STATE + 1,
    verbose=False
)

# Random Forest for diversity
models['rf'] = RandomForestAlgorithm(
    n_estimators=300,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Extra Trees for additional diversity
models['et'] = ExtraTreesAlgorithm(
    n_estimators=300,
    max_depth=10,
    min_samples_split=15,
    min_samples_leaf=8,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print(f"Created {len(models)} optimized models")

## 7. Cross-Validation and Model Training

In [ ]:
# Set up cross-validation
n_splits = 5
kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

# Store OOF predictions and test predictions
oof_predictions = {}
test_predictions = {}
cv_scores = {}

# Train each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    oof_pred = np.zeros(len(X_train))
    test_pred = np.zeros(len(X_test))
    fold_scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
        # Split data
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Train model
        model.fit(X_tr, y_tr)
        
        # Validate
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_pred[val_idx] = val_pred
        
        # Score
        val_score = accuracy_score(y_val, (val_pred > 0.5).astype(int))
        fold_scores.append(val_score)
        print(f"  Fold {fold_idx + 1}: {val_score:.4f}")
        
        # Test prediction
        test_pred += model.predict_proba(X_test)[:, 1] / n_splits
    
    # Store results
    oof_predictions[name] = oof_pred
    test_predictions[name] = test_pred
    cv_scores[name] = fold_scores
    
    # Overall CV score
    overall_score = accuracy_score(y_train, (oof_pred > 0.5).astype(int))
    print(f"  Overall CV: {overall_score:.4f} (+/- {np.std(fold_scores):.4f})")

## 8. Advanced Ensemble Strategies

In [ ]:
print("\n" + "="*50)
print("ENSEMBLE CREATION")
print("="*50)

ensemble_predictions = {}

# 1. Simple Average
print("\n1. Simple Average Ensemble")
simple_avg_oof = np.mean(list(oof_predictions.values()), axis=0)
simple_avg_test = np.mean(list(test_predictions.values()), axis=0)
simple_score = accuracy_score(y_train, (simple_avg_oof > 0.5).astype(int))
print(f"   CV Score: {simple_score:.4f}")
ensemble_predictions['simple_avg'] = simple_avg_test

# 2. Weighted Average based on CV scores
print("\n2. CV-Weighted Average Ensemble")
# Calculate weights based on CV performance
weights = {}
for name in cv_scores:
    weights[name] = np.mean(cv_scores[name])

# Normalize weights
total_weight = sum(weights.values())
weights = {k: v/total_weight for k, v in weights.items()}
print(f"   Weights: {weights}")

weighted_avg_oof = np.zeros(len(X_train))
weighted_avg_test = np.zeros(len(X_test))
for name in oof_predictions:
    weighted_avg_oof += oof_predictions[name] * weights[name]
    weighted_avg_test += test_predictions[name] * weights[name]

weighted_score = accuracy_score(y_train, (weighted_avg_oof > 0.5).astype(int))
print(f"   CV Score: {weighted_score:.4f}")
ensemble_predictions['weighted_avg'] = weighted_avg_test

# 3. Optuna-Optimized Weights
print("\n3. Optuna-Optimized Ensemble")
try:
    def objective(trial):
        # Suggest weights for each model
        w = {}
        for name in oof_predictions:
            w[name] = trial.suggest_float(f'weight_{name}', 0, 1)
        
        # Normalize weights
        total = sum(w.values())
        if total == 0:
            return 0
        w = {k: v/total for k, v in w.items()}
        
        # Calculate weighted prediction
        weighted_pred = np.zeros(len(X_train))
        for name in oof_predictions:
            weighted_pred += oof_predictions[name] * w[name]
        
        # Return accuracy
        return accuracy_score(y_train, (weighted_pred > 0.5).astype(int))
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100, show_progress_bar=False)
    
    # Get best weights
    best_params = study.best_params
    optuna_weights = {}
    for name in oof_predictions:
        optuna_weights[name] = best_params[f'weight_{name}']
    
    # Normalize
    total = sum(optuna_weights.values())
    optuna_weights = {k: v/total for k, v in optuna_weights.items()}
    print(f"   Optuna weights: {optuna_weights}")
    
    # Apply weights
    optuna_oof = np.zeros(len(X_train))
    optuna_test = np.zeros(len(X_test))
    for name in oof_predictions:
        optuna_oof += oof_predictions[name] * optuna_weights[name]
        optuna_test += test_predictions[name] * optuna_weights[name]
    
    optuna_score = accuracy_score(y_train, (optuna_oof > 0.5).astype(int))
    print(f"   CV Score: {optuna_score:.4f}")
    ensemble_predictions['optuna'] = optuna_test
    
except Exception as e:
    print(f"   Optuna optimization failed: {e}")

# 4. Stacking Ensemble
print("\n4. Stacking Ensemble")
try:
    # Prepare stacking features
    stack_features = np.column_stack(list(oof_predictions.values()))
    stack_test_features = np.column_stack(list(test_predictions.values()))
    
    # Train meta-model
    meta_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    meta_model.fit(stack_features, y_train)
    
    # Predictions
    stacking_oof = meta_model.predict_proba(stack_features)[:, 1]
    stacking_test = meta_model.predict_proba(stack_test_features)[:, 1]
    
    stacking_score = accuracy_score(y_train, (stacking_oof > 0.5).astype(int))
    print(f"   CV Score: {stacking_score:.4f}")
    ensemble_predictions['stacking'] = stacking_test
    
except Exception as e:
    print(f"   Stacking failed: {e}")

# 5. Rank Average
print("\n5. Rank Average Ensemble")
from scipy.stats import rankdata

rank_oof = np.zeros(len(X_train))
rank_test = np.zeros(len(X_test))

for name in oof_predictions:
    rank_oof += rankdata(oof_predictions[name])
    rank_test += rankdata(test_predictions[name])

rank_oof /= len(oof_predictions)
rank_test /= len(test_predictions)

# Normalize to [0, 1]
rank_oof = (rank_oof - rank_oof.min()) / (rank_oof.max() - rank_oof.min())
rank_test = (rank_test - rank_test.min()) / (rank_test.max() - rank_test.min())

rank_score = accuracy_score(y_train, (rank_oof > 0.5).astype(int))
print(f"   CV Score: {rank_score:.4f}")
ensemble_predictions['rank_avg'] = rank_test

## 9. Select Best Ensemble and Create Submission

In [ ]:
print("\n" + "="*50)
print("ENSEMBLE PERFORMANCE SUMMARY")
print("="*50)

# Compare all ensemble methods
ensemble_scores = {
    'Simple Average': simple_score,
    'CV-Weighted': weighted_score,
}

if 'optuna' in ensemble_predictions:
    ensemble_scores['Optuna-Optimized'] = optuna_score

if 'stacking' in ensemble_predictions:
    ensemble_scores['Stacking'] = stacking_score

if 'rank_avg' in ensemble_predictions:
    ensemble_scores['Rank Average'] = rank_score

# Sort by score
sorted_ensembles = sorted(ensemble_scores.items(), key=lambda x: x[1], reverse=True)

print("\nEnsemble Rankings:")
for i, (name, score) in enumerate(sorted_ensembles, 1):
    print(f"{i}. {name}: {score:.4f}")

# Select best ensemble
best_ensemble_name = sorted_ensembles[0][0]
print(f"\nBest ensemble: {best_ensemble_name} with CV score: {sorted_ensembles[0][1]:.4f}")

## 10. Generate Multiple Submissions

In [ ]:
# Create submission for each ensemble method
print("\n" + "="*50)
print("GENERATING SUBMISSIONS")
print("="*50)

# Map ensemble names to predictions
ensemble_map = {
    'Simple Average': 'simple_avg',
    'CV-Weighted': 'weighted_avg',
    'Optuna-Optimized': 'optuna',
    'Stacking': 'stacking',
    'Rank Average': 'rank_avg'
}

# Generate submissions
for ensemble_name, pred_key in ensemble_map.items():
    if pred_key in ensemble_predictions:
        # Create submission
        submission_df = pd.DataFrame({
            'PassengerId': test_df['PassengerId'],
            'Transported': (ensemble_predictions[pred_key] > 0.5).astype(bool)
        })
        
        # Save
        filename = f'submission_{pred_key}_v12.csv'
        submission_df.to_csv(filename, index=False)
        print(f"Saved: {filename}")

# Also save the best ensemble
best_pred_key = ensemble_map[best_ensemble_name]
best_submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': (ensemble_predictions[best_pred_key] > 0.5).astype(bool)
})
best_submission.to_csv('submission_best_v12.csv', index=False)
print(f"\nSaved best submission: submission_best_v12.csv")

## 11. Performance Summary

In [ ]:
print("\n" + "="*50)
print("FINAL PERFORMANCE SUMMARY - v12")
print("="*50)

print("\n📊 Model Performance:")
for name in models.keys():
    if name in cv_scores:
        scores = cv_scores[name]
        print(f"   {name}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

print("\n🎯 Ensemble Performance:")
for name, score in sorted_ensembles:
    print(f"   {name}: {score:.4f}")

print("\n✅ Key Improvements in v12:")
print("   - Advanced feature engineering (cabin, group, spending features)")
print("   - Optimized hyperparameters from baseline analysis")
print("   - Multiple ensemble strategies with Optuna optimization")
print("   - Smart missing value imputation")
print("   - Interaction and derived features")

print(f"\n🎉 Best CV Score: {sorted_ensembles[0][1]:.4f}")
print("\nTarget: Approach baseline's 82.066% accuracy")
print("\nAll submissions saved. Submit the best performing one to Kaggle!")